In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))
#print(PROJECT_ROOT)
#print(sys.path.insert(0, str(PROJECT_ROOT)))

In [2]:
from src.config.spark_session import get_spark_session

spark = get_spark_session()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/23 22:13:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
from pyspark.sql.functions import (
    col, when, count, to_timestamp, year, month, dayofmonth, dayofweek, isnan
)
from pyspark.sql.window import Window


In [4]:
data_path = "/Users/yassineoc/Desktop/hamza/Projet_Spark/Customer Segmentation and Churn Prediction with PySpark/data/raw/Online Retail.csv"

df = spark.read.csv(
    data_path,
    header=True,
    inferSchema=True
)

In [8]:

df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [13]:
# Calcul de la valeur la plus fréquente pour la description (mode)
mode_description = (
    df.groupBy("Description")
    .count()
    .orderBy(col("count").desc())
    .first()[0]
)

df_clean = df.fillna({"Description": mode_description})

#Suppression des lignes avec CustomerID manquant
df_clean = df.filter(col("CustomerID").isNotNull())

#print(df_clean)
df_clean.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df_clean.columns]).show()



+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|          0|       0|          0|        0|         0|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [14]:
# Création de la variable TotalPrice

df_clean = df_clean.withColumn(
    "TotalPrice",
    col("Quantity") * col("UnitPrice")
)


In [18]:
print(df_clean.select('InvoiceDate').show(10))

+--------------+
|   InvoiceDate|
+--------------+
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:28|
|12/1/2010 8:28|
|12/1/2010 8:34|
+--------------+
only showing top 10 rows

None


In [19]:
# Conversion et enrichissement de InvoiceDate

df_clean = df_clean.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "MM/dd/yyyy HH:mm")
)

df_clean = (
    df_clean
    .withColumn("Year", year(col("InvoiceDate")))
    .withColumn("Month", month(col("InvoiceDate")))
    .withColumn("Day", dayofmonth(col("InvoiceDate")))
    .withColumn("DayOfWeek", dayofweek(col("InvoiceDate")))
)


In [20]:
df_clean.printSchema()
df_clean.show(5)

print("Nombre de lignes après nettoyage :", df_clean.count())


root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- TotalPrice: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+----+-----+---+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|        TotalPrice|Year|Month|Day|DayOfWeek|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+----+-----+---+---------+
|   5363

Nombre de lignes après nettoyage : 406829


In [21]:
output_path = "/Users/yassineoc/Desktop/hamza/Projet_Spark/Customer Segmentation and Churn Prediction with PySpark/data/processed/online_retail_cleaned"

df_clean.write.mode("overwrite").parquet(output_path)

25/12/24 05:25:34 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 975353 ms exceeds timeout 120000 ms
25/12/24 05:25:34 WARN SparkContext: Killing executors is not supported by current scheduler.
25/12/24 05:25:38 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$